# 01 — Bronze Layer: Raw Data Ingestion

This notebook demonstrates the **Bronze layer** of the B3 Data Platform.

**Goal:** ingest raw daily OHLCV prices from Yahoo Finance and persist them as-is
(no transformation) into the Bronze Parquet store.

Architecture reminder:
```
Yahoo Finance API  ──►  Bronze (raw Parquet)  ──►  Silver  ──►  Gold
```
**Rules for Bronze:**
- Data is written exactly as received — no value changes.
- Only metadata columns are added: `source` and `ingested_at`.
- Partitioned by `trade_date` for fast downstream reads.

In [ ]:
import sys
sys.path.insert(0, "..")

from datetime import date, timedelta

import polars as pl

pl.Config.set_tbl_rows(20)
print(f"Polars version: {pl.__version__}")

## 1. Fetch raw prices from Yahoo Finance

In [ ]:
from c_ingestion.yahoo_finance import fetch_daily_prices

END   = date.today()
START = END - timedelta(days=90)  # last 3 months

# Use a small subset for the demo
DEMO_TICKERS = ["PETR4.SA", "VALE3.SA", "ITUB4.SA", "BBDC4.SA", "ABEV3.SA"]

raw_df = fetch_daily_prices(tickers=DEMO_TICKERS, start=START, end=END)
print(f"Fetched {len(raw_df):,} rows for {raw_df['ticker'].n_unique()} tickers")
raw_df.head(10)

## 2. Inspect raw data

In [ ]:
print("Schema:")
print(raw_df.schema)
print("\nNull counts:")
print(raw_df.null_count())
print("\nBasic stats:")
raw_df.describe()

In [ ]:
# Date coverage per ticker
raw_df.group_by("ticker").agg(
    pl.col("trade_date").min().alias("first_date"),
    pl.col("trade_date").max().alias("last_date"),
    pl.col("trade_date").n_unique().alias("trading_days"),
).sort("ticker")

## 3. Write to Bronze layer

In [ ]:
from d_processing.bronze.ingest import write_bronze

bronze_path = write_bronze(raw_df, source="yahoo_finance")
print(f"Written to: {bronze_path}")

## 4. Read back from Bronze and verify

In [ ]:
from d_processing.bronze.ingest import read_bronze

bronze_df = read_bronze(source="yahoo_finance")
print(f"Bronze rows: {len(bronze_df):,}")
print(f"Columns added: {[c for c in bronze_df.columns if c not in raw_df.columns]}")
bronze_df.head(10)

## 5. Quick visualization — closing prices

In [ ]:
import plotly.express as px

fig = px.line(
    bronze_df.sort("trade_date").to_pandas(),
    x="trade_date",
    y="close",
    color="ticker",
    title="B3 Closing Prices — Bronze Layer (raw)",
    labels={"trade_date": "Date", "close": "Close Price (BRL)", "ticker": "Ticker"},
)
fig.show()

## 6. Run full Bronze pipeline (all default tickers)

In [ ]:
from f_pipelines.bronze_pipeline import BronzePipeline, BronzePipelineConfig

cfg = BronzePipelineConfig(start=START, end=END)
pipeline = BronzePipeline(config=cfg)
result = pipeline.run()
print(f"Pipeline produced {len(result):,} rows")